In [ ]:
import copy
import logging
import numpy as np
import pymoo.model.crossover
import pymoo.model.mutation
import pymoo.model.problem
import pymoo.model.repair
import pymoo.optimize

from collections import OrderedDict
from pymoo.algorithms.nsga2 import NSGA2
from pymoo.operators.crossover.util import crossover_mask
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

import matplotlib.pyplot as plt

import math
from enum import Enum
import os
import sys
import yaml
BLOCK_SIZE = 16
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))
from benchmark_test.analysis.profile.profile import InstanceDecodeProfiler, InstanceDecodeProfilerV2, InstanceProfile, MigrationProfile


# 1. 建模

In [ ]:
class PrefillRateModel:
    def __init__(self, prompt_len, fit_args):
        self.prompt_len = prompt_len
        self.fit_args = fit_args

    def get_time(self, token_num=None):
        latency_args = self.fit_args
        # y = a*x^2 + b*x + c
        return latency_args[0] * token_num**2 + latency_args[1] * token_num + latency_args[2]
    
    def get_deal_max_rare(self):
        # 基于导数得到的处理速率，适用于高并发场景。实际上可能会长时间处于低并发场景
        latency_args = self.fit_args
        avg_token_num = self.get_time(self.in_rate * self.prompt_len) / 1000 * (self.in_rate * self.prompt_len) # 平均token数

        args = [2*latency_args[0], latency_args[1]] # 导数
        # print(args[0] * self.prompt_len + args[1], args[1])
        # return (1000 / (args[0] * avg_token_num + args[1])) / self.prompt_len
        return (1000 / (args[0] * avg_token_num + args[1])) / self.prompt_len
    
    def get_deal_rare(self, qps):
        # TODO 如何考虑请求batch到一起的情况(利用qps)
        args = self.fit_args
        return 1000 / (args[0] * self.prompt_len**2 + args[1]*self.prompt_len + args[2])
    
    def get_out_rate(self, qps=None):
        return self.get_deal_rare(qps)
    
class DecodeRateModelV2:
    def __init__(self, model_config, workload_msg, fit_args):
        self.model_config = model_config
        self.prompt_len = workload_msg['prompt_len']
        self.output_len = workload_msg['output_len']
        # print(fit_args)
        self.fit_args = fit_args

    def solve_batch_size(self, lam):
        """
        求解 batch_size 的显式表达式

        参数:
            l_output: 输出层维度
            k2: 权重系数
            seq_len: 序列长度
            c: 常数项
            k1: 权重系数
            lam: 流量系数 λ

        返回:
            batch_size 的两个可能解（通常选择正解）
        """

        mu_d = self.solve_mu_d(lam)
        bs = lam / (mu_d - lam)
        return bs

    def solve_mu_d(self, lam):
        """
        求解μ_d的有效速率参数
        
        参数:
        l_output : float   输出层维度
        k2       : float   权重系数
        seq_len  : float   序列长度
        c        : float   常数项
        k1       : float   权重系数
        lam      : float   流量系数
        
        返回:
        mu_d1 : μ_d解
        """
        l_output = self.output_len
        k2 = self.fit_args[1]
        seq_len = self.get_seq_len()
        c = self.fit_args[2]
        k1 = self.fit_args[0]

        
        # 计算二次项系数
        A = l_output * c
        B = lam * l_output * (-c + k1 + k2 * seq_len)
        C = -1000 * lam             # 由于其余单位均基于ms，所以lam需乘以1000

        # 判别式计算
        discriminant = B**2 - 4 * A * C
        print(k1, k2, c, l_output * (-c + k1 + k2 * seq_len))
        print(f"lam={lam}, B={B}, A={A}, C={C}, discriminant={discriminant}, {math.sqrt(discriminant)}")

        if discriminant < 0:
            print(f"lam={lam}, B={B}, A={A}, C={C}")
            print(f"B**2 - 4 * A * C = {discriminant} < 0. Can not get root")
            return 0
        
        # 求解公式
        denominator = 2 * A
        
        # 计算两个解
        mu_d1 = (-B + math.sqrt(discriminant)) / denominator
        mu_d2 = (-B - math.sqrt(discriminant)) / denominator
        print(f"mu_d solutions for λ={lam}: {mu_d1}, {mu_d2}")
        return mu_d1
    
    def get_seq_len(self):
        return self.prompt_len + 0.7 * self.output_len
    
    def get_max_batch_size(self):
        gpu_block_num = self.model_config['gpu_block']
        return gpu_block_num * BLOCK_SIZE / self.get_seq_len()
        # return gpu_block_num * BLOCK_SIZE / (self.prompt_len + self.output_len)
    
    # 根据请求处理速度计算batch_size
    # qps: 请求处理速度（mu_d），单位req/s
    def get_batch_size_according_to_mu_d(self, qps=None):
        if qps is None:
            qps = self.in_rate
        args = self.fit_args
        batch_size = (qps * self.output_len * args[2]) / \
                    (1000 - qps * self.output_len * (args[0] + args[1] * self.get_seq_len()))
        return batch_size
    
    # 根据batch_size计算处理速度
    def get_out_rate_according_to_batch_size(self, batch_size=None):
        if batch_size is None:
            batch_size = self.get_max_batch_size()
        args = self.fit_args
        rate = 1000 * batch_size / ((args[0] * batch_size + args[1] * self.get_seq_len() * batch_size + args[2]) * self.output_len)
        return rate
    
    # 根据请求到达速度计算处理速度(适用于实际情况)
    def get_out_rate(self, qps=None):
        mu_d = self.solve_mu_d(qps)
        # 选择正实数解
        # print(f"qps: {qps}, mu_d solutions: {mu_d}, batch_size(assume mu_d=qps): {self.get_batch_size_according_to_mu_d(qps)}, batch_size:{self.get_batch_size_according_to_mu_d(mu_d)}")
        return mu_d
    
    # 根据请求处理速度计算batch_size(适用于实际情况)
    def get_batch_size_according_to_qps(self, qps=None):
        bs = self.solve_batch_size(qps)
        return bs
    
    # 根据batch_size计算解码时间
    def get_decode_time(self, batch_size=None, seq_len=None):
        args = self.fit_args
        if seq_len is None:
            seq_len = self.get_seq_len()
        return (args[0] * batch_size + args[1] * self.get_seq_len() * batch_size + args[2])

class MigrationRateModel:
    def __init__(self, prompt_len, fit_args):
        self.prompt_len = prompt_len
        self.fit_args = fit_args
        
    def get_rate(self):
        args = self.fit_args
        migrate_time = (args[0] * self.prompt_len / BLOCK_SIZE + args[1]) + 50
        # print(f'Migration time : {migrate_time}')
        return 1000 / migrate_time
    
class DisaggregationRateModel:
    def __init__(self, model_profiler_msg, workload_msg, placement):
        prefill_tps, decode_tps = placement
        self.is_pd = True

        self.qps = workload_msg['qps']
        self.output_len = workload_msg['output_len']
        self.prompt_len = workload_msg['prompt_len']

        self.prefill_tps = prefill_tps
        self.decode_tps = decode_tps

        self.prefill_models = []
        self.decode_models = []
        self.prefill_out_rates = []
        for tp in prefill_tps:
            prefill_model = PrefillRateModel(workload_msg['prompt_len'], model_profiler_msg[tp]['prefill_msg'])
            self.prefill_models.append(prefill_model)
            # self.prefill_out_rates.append(prefill_model.get_out_rate())
        
        from_tp = prefill_tps[0]
        to_tp = decode_tps[0]
        # TODO 目前只考虑单个迁移节点
        # print(f"Migration from {from_tp} to {to_tp}")
        # print(f"Migration model args: {model_profiler_msg}")
        if 'migrateion_rate' in model_profiler_msg:
            self.migrateion_rate = model_profiler_msg['migrateion_rate']
        else:
            self.migration_model = MigrationRateModel(workload_msg['prompt_len'], model_profiler_msg[from_tp]['migrate_msg'][to_tp])
            self.migrateion_rate = self.migration_model.get_rate() * min(len(prefill_tps), len(decode_tps))

        for tp in decode_tps:
            decode_model = DecodeRateModelV2(
                model_profiler_msg[tp]['model_msg'], 
                workload_msg, 
                model_profiler_msg[tp]['decode_msg']
            )
            self.decode_models.append(decode_model)
    def get_prefill_qps_list(self):
        # 根据总qps，计算各个prefill模型的qps。（按照GPU数划分）
        qps_list = []
        gpu_sum = sum(self.prefill_tps)
        for tp in self.prefill_tps:
            qps_list.append(self.qps * tp / gpu_sum)
        return qps_list
    
    def get_prefill_rate(self, qps=None):
        if qps is None:
            prefill_rates = [pm.get_out_rate() for pm in self.prefill_models]
        else:
            prefill_rates = []
            for prefill_model, q in zip(self.prefill_models, qps):
                prefill_rates.append(prefill_model.get_out_rate(q))
        return prefill_rates 
    
    def get_decode_rate_according_to_bs(self, batch_size=None):
        total_rate = 0
        if batch_size is None:
            batch_size = [dm.get_max_batch_size() for dm in self.decode_models]
        for decode_model, bs in zip(self.decode_models, batch_size):
            total_rate += decode_model.get_out_rate_according_to_batch_size(bs)
        return total_rate  
    
    def get_decode_in_rate(self):
        # print(f"Decode_in_rate: Min(Sum prefill rates: {sum(self.get_prefill_rate())}, Migration rate: {self.migrateion_rate}, QPS: {self.qps})")
        return min(sum(self.get_prefill_rate()), self.migrateion_rate, self.qps)
    
    def get_decode_qps_list(self, qps=None):
        # 根据总qps，计算各个decode模型的qps。（按照GPU数划分）
        qps_list = []
        gpu_sum = sum(self.decode_tps)
        for tp in self.decode_tps:
            qps_list.append(qps * tp / gpu_sum)
        return qps_list

    def get_decode_rate(self, qps=None):
        decode_rates = []
        for decode_model, q in zip(self.decode_models, qps):
            decode_rates.append(decode_model.get_out_rate(q))
        return decode_rates   
    
    
    def get_rate_msg(self): # TODO
        prefill_qps_list = self.get_prefill_qps_list()
        prefill_rates = self.get_prefill_rate(prefill_qps_list)
        prefill_latency = [1000 / (prefill_rate - self.qps) if prefill_rate > self.qps else float('inf')
                           for prefill_rate in prefill_rates]
        fit_prefill_time = [dm.get_time(self.prompt_len) for dm in self.prefill_models]

        migration_rate = self.migrateion_rate

        decode_total_in_rate = self.get_decode_in_rate()
        decode_qps_list = self.get_decode_qps_list(decode_total_in_rate)        # 各个decode模型的qps
        decode_rates = self.get_decode_rate(decode_qps_list)                    # 各个decode模型的处理速率
        decode_latency = [1000 / (decode_deal_rate - decode_in_rate) / self.output_len if decode_deal_rate > decode_in_rate else float('inf')
                          for decode_deal_rate, decode_in_rate in zip(decode_rates, decode_qps_list)]
        fit_batch_size = [dm.get_batch_size_according_to_qps(q) for dm, q in zip(self.decode_models, decode_qps_list)] # 各个decode模型的batch_size（fit）
        max_batch_size = [dm.get_max_batch_size() for dm in self.decode_models]
        fit_decode_time = [dm.get_decode_time(bs) for dm, bs in zip(self.decode_models, fit_batch_size)]
        return {
            'prefill_qps_list': prefill_qps_list,
            'prefill_rate': prefill_rates,
            'prefill_latency': prefill_latency,
            'fit_prefill_time': fit_prefill_time,
            'migration_rate': migration_rate,
            'decode_in_rate': decode_qps_list,
            'decode_rate': decode_rates,
            'decode_latency' : decode_latency,
            'fit_batch_size': fit_batch_size,
            'fit_decode_time': fit_decode_time,
            'max_batch_size': max_batch_size,
            'max_decode_rate': [dm.get_out_rate_according_to_batch_size() for dm in self.decode_models],
        }

class AggregationRateModelBaseV3:
    def __init__(self, model_profiler_msg, workload_msg, tp):
        self.prompt_len = workload_msg['prompt_len']
        self.output_len = workload_msg['output_len']
        self.fit_prefill_args = model_profiler_msg[tp]['prefill_msg']
        self.fit_decode_args = model_profiler_msg[tp]['decode_msg']

        gpu_block_num = model_profiler_msg[tp]['model_msg']['gpu_block']
        self.max_batch_size = gpu_block_num * BLOCK_SIZE / self.get_seq_len()
        self.max_seq_len = model_profiler_msg[tp]['model_msg']['max_seq_len']
    
    def solve_mu(self, lam):
        prompt_len = self.prompt_len
        output_len = self.output_len
        seq_len = self.get_seq_len()
        a = self.fit_prefill_args[0]
        # a = 0.0
        b = self.fit_prefill_args[1]
        c = self.fit_prefill_args[2]

        k1 = self.fit_decode_args[0]
        k2 = self.fit_decode_args[1]
        k3 = self.fit_decode_args[2]
        A = k3 * output_len
        B = lam * (a * prompt_len**2 + b * prompt_len + c + (k1 + k2 * seq_len) * output_len - k3 * output_len)
        C = -1000 * lam
        print(k1, k2, k3, (k1 + k2 * seq_len-k3)* output_len)
        print(a * prompt_len**2 + b * prompt_len + c, (k1 + k2 * seq_len-k3)* output_len)

        # 判别式计算
        discriminant = B**2 - 4 * A * C
        
        # 求解公式
        denominator = 2 * A
        
        # 计算两个解
        mu1 = (-B + math.sqrt(discriminant)) / denominator
        mu2 = (-B - math.sqrt(discriminant)) / denominator
        # print(f"λ={lam}, mu solutions: {mu1}, {mu2}")
        return [mu1]

    def solve_batch_size(self, lam):
        prompt_len = self.prompt_len
        output_len = self.output_len
        seq_len = self.get_seq_len()
        a = self.fit_prefill_args[0]
        # a = 0.0
        b = self.fit_prefill_args[1]
        c = self.fit_prefill_args[2]

        k1 = self.fit_decode_args[0]
        k2 = self.fit_decode_args[1]
        k3 = self.fit_decode_args[2]

        s = a * prompt_len**2 + b * prompt_len + c + (k1 + k2 * seq_len) * output_len

        A = -(lam * s - 1000)
        B = -(lam * s + lam * k3 * output_len)
        C = -(lam * k3 * output_len)

        # 判别式计算
        discriminant = B**2 - 4 * A * C

        if discriminant < 0:
            print("B**2 - 4 * A * C < 0. Can not get root")
            return [0]
        
        # 求解公式
        denominator = 2 * A
        
        # 计算两个解
        bs1 = (-B + math.sqrt(discriminant)) / denominator
        bs2 = (-B - math.sqrt(discriminant)) / denominator
        # print(f"λ={lam}, bs solutions: {bs1}, {bs2}")
        return [bs1]

    def get_prefill_time(self):
        # y = a*x^2 + b*x + c
        latency_args = self.fit_prefill_args
        time = 0
        token_num = self.prompt_len
        while token_num > self.max_seq_len:
            time += latency_args[0] * self.max_seq_len**2 + latency_args[1] * self.max_seq_len + latency_args[2]
            token_num -= self.max_seq_len
        # 计算剩余token_num的时间
        time += latency_args[0] * token_num**2 + latency_args[1] * token_num + latency_args[2]
        return time
    
    def get_seq_len(self):
        return self.prompt_len + 0.5 * self.output_len

    def get_decode_time(self, batch_size=None):
        if batch_size is None:
            batch_size = self.get_max_batch_size()
        args = self.fit_decode_args
        time = (args[0] * batch_size + args[1] * self.get_seq_len() * batch_size + args[2]) * self.output_len
        return time
    
    def get_batch_size(self,qps):
        return self.solve_batch_size(qps)[0]
    
    # def get_out_rate(self, batch_size):
    #     prefill_time = self.get_prefill_time(batch_size * self.prompt_len)
    #     rate = batch_size / (prefill_time / 1000 + self.get_decode_time(batch_size) / 1000)
    #     return rate
    def get_out_rate(self, qps):
        # rate = self.solve_mu_p(qps)
        rate = self.solve_mu(qps)
        print(rate)
        bs = min(self.solve_batch_size(qps))
        rate = qps + qps / bs
        print(rate)
        return rate
    
    def get_time(self, bs):
        prefill_time = bs * self.get_prefill_time()
        decode_time = self.get_decode_time(bs)
        return (prefill_time + decode_time) / bs


class AggregationRateModel:
    def __init__(self, model_profiler_msg, workload_msg, placement):
        assert len(placement[1]) == 0
        self.is_pd = False
        tps = placement[0]
        self.tps = tps
        self.qps = workload_msg['qps']
        self.output_len = workload_msg['output_len']

        self.models = []
        for tp in tps:
            self.models.append(AggregationRateModelBaseV3(model_profiler_msg, workload_msg, tp))
    def get_out_rate(self, batch_size=None):
        total_rate = 0
        for model,bs in zip(self.models, batch_size):
            total_rate += model.get_out_rate(bs)
        return total_rate
    def get_qps_list(self, qps=None):
        # 根据总qps，计算各个decode模型的qps。（按照GPU数划分）
        qps_list = []
        gpu_sum = sum(self.tps)
        for tp in self.tps:
            qps_list.append(qps * tp / gpu_sum)
        return qps_list

    def get_rate(self, qps=None):
        rates = []
        for model, q in zip(self.models, qps):
            rates.append(model.get_out_rate(q))
        return rates 
    
    def get_rate_msg(self): # TODO
        qps_list = self.get_qps_list(self.qps)
        rates = self.get_rate(qps_list)
        # print(rates)
        latency = [1000 / (rate - qps) if rate > qps else float('inf')
                           for rate,qps in zip(rates, qps_list)]
        fit_batch_size = [m.get_batch_size(q) for m, q in zip(self.models, qps_list)] # 各个模型的batch_size（fit）
        # print(fit_batch_size)
        max_batch_size = [m.max_batch_size for m in self.models]
        fit_all_decode_time = [m.get_decode_time(bs) for m, bs in zip(self.models, fit_batch_size)]
        return {
            'qps_list': qps_list,
            'rate': rates,
            'latency': [l / 1000 for l in latency], # s 单位
            'fit_batch_size': fit_batch_size,
            'fit_all_decode_time': [t / 1000 for t in fit_all_decode_time], # s 单位
            'fit_decode_time': [t/self.output_len for t in fit_all_decode_time],
            'max_batch_size': max_batch_size,
            'fit_time': [m.get_time(bs) for m, bs in zip(self.models, fit_batch_size)],
        }
    
class ModelProfiler:
    def __init__(self, model_name, profile_dir, tp, cover=False):
        self.model_name = model_name
        self.profile_dir = profile_dir
        self.tp = tp 
        self.profile_data = {} # [migrate_msg,prefill,decode,model_msg[max_seq_len,gpu_block,cpu_block]]
        with open('config_model.yaml', 'r', encoding='utf-8') as file:
            self.model_config = yaml.safe_load(file)

        if os.path.exists('profile_output.yaml'):
            with open('profile_output.yaml', 'r', encoding='utf-8') as file:
                self.all_profile_data = yaml.safe_load(file) 
        else:
            self.all_profile_data = {}
        if not cover and self.model_name in self.all_profile_data and self.tp in self.all_profile_data[self.model_name]:
            self.profile_data = self.all_profile_data[self.model_name]
        else:
            self.get_model_profile()
    
    def get_model_profile(self):
        self.get_model_msg()
        self.get_migrate_msg()
        self.get_prefill_msg()
        self.get_decode_msg()
        self.save_profile(f'profile_output.yaml')
    
    def save_profile(self, save_path):
        if self.model_name not in self.all_profile_data:
            self.all_profile_data[self.model_name] = {}
        self.all_profile_data[self.model_name][self.tp] = self.profile_data
        with open(save_path, 'w', encoding='utf-8') as f:
            yaml.safe_dump_all(documents=[self.all_profile_data], stream=f, allow_unicode=True)

    def get_files_for_migrate_analysis(self, to_tp=None):
        dir = f'{self.profile_dir}/{self.model_name}/poisson/'
        # 列出dir下所有文件
        files = os.listdir(dir)
        # 过滤出包含tp的文件
        files = [f for f in files if f'_pdd_' in f]

        new_files = []
        for f in files:
            tp_list = f.split('_')[4]
            if f'{self.tp}-{to_tp}' in tp_list:
                new_files.append(f)
        new_files = [f for f in new_files if f'.log' in f]
        new_files = [f'{dir}/{f}' for f in new_files]
        # assert len(new_files) > 0, f'No profile files found in {dir} for tp {self.tp}'
        return new_files
    
    def get_migrate_msg(self):
        concurrency = 1
        # file = [
        #     f'{self.profile_dir}/{self.model_name}/poisson/serve_pdd_tp1_2000_qps_6_1_3.log',
        #     f'{self.profile_dir}/{self.model_name}/poisson/serve_pdd_tp1_2000_qps_2_1_3.log',
        #     f'{self.profile_dir}/{self.model_name}/poisson/serve_pdd_tp1_2000_qps_4_1_3.log',
        # ]
        for tp in [1, 2, 4]:
            file = self.get_files_for_migrate_analysis(tp)
            if len(file) == 0:
                continue
            # print(f"Migration files: {file}")
            migration_profile = MigrationProfile(file, 0.8)
            migration_profile.get_migration_info()
            if 'migrate_msg' not in self.profile_data:
                self.profile_data['migrate_msg'] = {}
            self.profile_data['migrate_msg'][tp] = [float(x) for x in migration_profile.fit_results]
    
    def get_prefill_msg(self):
        profile_dir = f'{self.profile_dir}/{self.model_name}/poisson'
        prefill_profile = InstanceProfile(profile_dir, self.tp, enable_pd=True, is_verbose=True)
        prefill_profile.get_pdd_instance_info()
        self.profile_data['prefill_msg'] = [float(x) for x in prefill_profile.fit_results['prefill']]
    
    def get_decode_msg(self):
        profile_dir = f'{self.profile_dir}/{self.model_name}/poisson'
        decode_profile = InstanceDecodeProfilerV2(profile_dir, self.tp, enable_pd=True, is_verbose=True)
        decode_profile.get_pdd_instance_info()
        self.profile_data['decode_msg'] = [float(decode_profile.model.coef_[0]), float(decode_profile.model.coef_[1]), float(decode_profile.model.intercept_)]
    def get_model_msg(self):
        self.profile_data['model_msg'] = {
            'max_seq_len': self.model_config[self.model_name]['max_seq_len'],
            'gpu_block': self.model_config[self.model_name][self.tp]['gpu_block'],
            'cpu_block': self.model_config[self.model_name][self.tp]['cpu_block'],
        }

# 2. 优化
目前只考虑同个节点PD分离的情况

In [ ]:
class InstanceType(int, Enum):
    PREFILL = 0
    DECODE = 1
    NO_CONSTRAINTS = 2

prompt_lens = {
    'llama-7b': 182,
    'llama-13b': 261,
}
output_lens = {
    'llama-7b': 473,
    'llama-13b': 491,
}


## 2.1 Problem

In [ ]:
class Crossover(pymoo.model.crossover.Crossover):
    def __init__(self):
        '''
            n_parents: 每次交叉操作需要两个父代（即从两个分配方案中“混合”生成后代）
            n_offsprings: 每次交叉操作会生成两个后代
        '''
        super().__init__(n_parents=2, n_offsprings=2)

    def _do(self, problem, states, **kwargs):
        return problem._crossover(states, **kwargs)


class Mutation(pymoo.model.mutation.Mutation):
    def _do(self, problem, states, **kwargs):
        return problem._mutation(states, **kwargs)


class Repair(pymoo.model.repair.Repair):
    def _do(self, problem, pop, **kwargs):
        return problem._repair(pop, **kwargs)

class Problem(pymoo.model.problem.Problem):
    def __init__(self, model, workload_msg, base_state, is_verbose=False):
        self._base_state = base_state

        max_gpus = base_state.shape[0]
        self._num_nodes = base_state.shape[1] - 1
        self._num_gpus = max_gpus // self._num_nodes

        self._model = model
        if workload_msg['is_ShareGPT']:
            workload_msg['prompt_len'] = prompt_lens[model]
            workload_msg['output_len'] = output_lens[model]
        self._workload_msg = workload_msg
        self.model_profiler_msg = None

        self.is_verbose = is_verbose

        super().__init__(n_var=self._base_state.size, n_obj=2, type_var=np.int)

    def _state_to_placement(self, state):
        '''
        Convert a cluster state to a placement configuration.
        param state: The state of the cluster.
        return: The placement configuration.
        '''
        # 暂时只考虑一个节点   只考虑一个节点: [(prefill_tps, decode_tps), (tps, [])]
        prefill_tps = []
        decode_tps = []
        tps = []
        for instance_idx in range(state.shape[0]):
            node_idx = np.where(state[instance_idx, 1:] > 0)[0]     # 找到该实例所在所有节点的索引
            node_gpus = state[instance_idx, 1:][node_idx]               # 找到该实例所在所有节点的GPU数
            if len(node_idx) == 0:
                continue
            if state[instance_idx, 0] == InstanceType.PREFILL:
                prefill_tps.append(sum(node_gpus))
            elif state[instance_idx, 0] == InstanceType.DECODE:
                decode_tps.append(sum(node_gpus))
            elif state[instance_idx, 0] == InstanceType.NO_CONSTRAINTS:
                tps.append(sum(node_gpus))
        return [(prefill_tps, decode_tps), (tps, [])]
    
    def get_model_profiler_msg(self, scale, profile_dir=None, concurrency=1):
        if profile_dir is None:
            profile_dir = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-2-concurrency-{concurrency}'
        m = ModelProfiler(self._model, profile_dir, 1)    # 这里假设已经profile好了，调用这个类只是获取数据
        if 'migrateion_rate' in scale:
            m.profile_data['migrateion_rate'] = scale['migrateion_rate']
        elif 'migrateion' in scale:
            for k,v in m.profile_data.items():
                for to_tp, arg in v['migrate_msg'].items():
                    m.profile_data[k]['migrate_msg'][to_tp][0] *= scale['migrateion']  # 对迁移拟合结果的斜率进行缩放
        self.model_profiler_msg = m.profile_data
    
    def get_results_msg(self, rate_model):
        msg = rate_model.get_rate_msg()
        # print(f"Placement: {instance_deploy_msg}, Rate msg: {msg}")
        for k,v in msg.items():
            if isinstance(v, list):
                # 是否所有值都相同
                if all(x == v[0] for x in v):
                    msg[k] = v[0]
                else:
                    msg[k] = np.array(v)
        return msg
    
    def _get_pd_model_result(self, workload_msg, pd_deployment_msg):
        rate_model = DisaggregationRateModel(self.model_profiler_msg, workload_msg, pd_deployment_msg)
        rate_msg = self.get_results_msg(rate_model)
        if self.is_verbose:
            print(f"Rate model message: {rate_msg}")
        if (
            rate_msg['migration_rate'] < workload_msg['qps']
            or (np.array(rate_msg['fit_batch_size']) > np.array(rate_msg['max_batch_size'])).any()
            or (np.array(rate_msg['fit_batch_size']) < 0).any()
        ):
            if self.is_verbose:
                print(f"Invalid placement: {pd_deployment_msg}, migration_rate: {rate_msg['migration_rate']}, qps: {self._workload_msg['qps']}")
            return 10e9   # 无效解
        else:
            migration_time = 1000 / rate_msg['migration_rate']
            # latency = rate_msg['fit_prefill_time'] + migration_time + rate_msg['fit_decode_time'] * self._workload_msg['output_len']
            # latency = (rate_msg['fit_batch_size'] * rate_msg['fit_prefill_time'] + migration_time + rate_msg['fit_decode_time'] * self._workload_msg['output_len']) / rate_msg['fit_batch_size']
            latency = rate_msg['fit_decode_time']
            return latency
        
    def _get_no_constraints_model_result(self, workload_msg, no_constraints_deployment_msg):
        rate_model = AggregationRateModel(self.model_profiler_msg, workload_msg, no_constraints_deployment_msg)
        rate_msg = self.get_results_msg(rate_model)
        if self.is_verbose:
            print(f"Rate model message: {rate_msg}")
        if (
            (np.array(rate_msg['fit_batch_size']) > np.array(rate_msg['max_batch_size'])).any()
            or (np.array(rate_msg['fit_batch_size']) < 0).any()
        ):
            if self.is_verbose:
                print(f"Invalid placement: {no_constraints_deployment_msg}, qps: {self._workload_msg['qps']}")
            return 10e9   # 无效解
        else:
            # latency = rate_msg['fit_time']
            latency = rate_msg['fit_decode_time']
            return latency

    def _get_model_result(self, model, workload_msg, instance_deploy_msg, concurrency=1, profile_dir=None, is_verbose=False):
        '''
        Get the model result for a specific workload and deployment configuration.
        param model: The model name.
        param workload_msg: The workload message.
        param instance_deploy_msg: The deployment configuration. [(prefill_tps, decode_tps), (tps, [])]

        '''
        if profile_dir is None:
            profile_dir = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-2-concurrency-{concurrency}'
        if self.model_profiler_msg is None:
            m = ModelProfiler(model, profile_dir, 1)    # 这里假设已经profile好了，调用这个类只是获取数据
            self.model_profiler_msg = m.profile_data         # model_profiler_msg 包含各个tp的profile数据

        if not workload_msg['is_ShareGPT']:
            prompt_len = workload_msg['prompt_len']
            output_len = workload_msg['output_len']
            profile_dir += f'-{prompt_len}-{output_len}'    

        pd_deployment_msg = instance_deploy_msg[0]  # (prefill_tps, decode_tps)
        prefill_tps = pd_deployment_msg[0]
        decode_tps = pd_deployment_msg[1]

        no_constraints_deployment_msg = instance_deploy_msg[1]  # (tps, [])
        tps = no_constraints_deployment_msg[0]

        # 分三种情况
        # 1. 只考虑prefill和decode
        if len(prefill_tps) > 0 and len(decode_tps) > 0 and len(tps) == 0:
            return self._get_pd_model_result(workload_msg, pd_deployment_msg)
        # 2. 只考虑tps
        elif len(tps) > 0 and len(prefill_tps) == 0 and len(decode_tps) == 0:
            return self._get_no_constraints_model_result(workload_msg, no_constraints_deployment_msg)
        # 3. 三种都考虑
        elif len(tps) > 0 and (len(prefill_tps) > 0 and len(decode_tps) > 0):
            # 先按照gpu数比例划分qps
            total_gpus = sum(tps) + sum(prefill_tps) + sum(decode_tps)
            pd_qps = self._workload_msg['qps'] * (sum(prefill_tps) + sum(decode_tps)) / total_gpus
            no_constraints_qps = self._workload_msg['qps'] * sum(tps) / total_gpus
            # 判断哪个模型的latency更大
            pd_workload_msg = self._workload_msg.copy()
            pd_workload_msg['qps'] = pd_qps
            no_constraints_workload_msg = self._workload_msg.copy()
            no_constraints_workload_msg['qps'] = no_constraints_qps

            pd_latency = self._get_pd_model_result(pd_workload_msg, pd_deployment_msg)
            no_constraints_latency = self._get_no_constraints_model_result(no_constraints_workload_msg, no_constraints_deployment_msg)
            
            counter = 0
            if pd_latency > no_constraints_latency:
                # 降低pd模型的qps，增加no_constraints模型的qps
                while abs(pd_latency - no_constraints_latency) > 10 and pd_latency > no_constraints_latency:
                    if self.is_verbose:
                        print(f"Before Adjust\n  qps, pd_qps: {pd_workload_msg['qps']}, no_constraints_qps: {no_constraints_workload_msg['qps']}")
                        print(f"  pd_latency: {pd_latency}, no_constraints_latency: {no_constraints_latency}\n")
                    # pd_workload_msg['qps'] = pd_qps * no_constraints_latency / pd_latency
                    pd_workload_msg['qps'] = pd_qps - 0.01
                    no_constraints_workload_msg['qps'] = self._workload_msg['qps'] - pd_workload_msg['qps']
                    if pd_workload_msg['qps'] < 0:
                        no_constraints_workload_msg['qps'] = self._workload_msg['qps']
                        pd_latency = 0
                        no_constraints_latency = self._get_no_constraints_model_result(no_constraints_workload_msg, no_constraints_deployment_msg)
                        print(f"{instance_deploy_msg}: pd_qps < 0, , pd_latency:{pd_latency}({pd_workload_msg['qps']}), no_constraints_latency:{no_constraints_latency}({no_constraints_workload_msg['qps']})")
                        break
                    pd_latency = self._get_pd_model_result(pd_workload_msg, pd_deployment_msg)
                    no_constraints_latency = self._get_no_constraints_model_result(no_constraints_workload_msg, no_constraints_deployment_msg)
                    pd_qps = pd_workload_msg['qps']
                    counter += 1
                    if self.is_verbose:
                        print(f"After Adjust\n  qps, pd_qps: {pd_workload_msg['qps']}, no_constraints_qps: {no_constraints_workload_msg['qps']}")
                        print(f"  pd_latency: {pd_latency}, no_constraints_latency: {no_constraints_latency}\n")
                    if counter > 2000:
                        print(f"Adjust more than {counter} times")
                        print(f"Adjusting qps, pd_qps: {pd_workload_msg['qps']}, no_constraints_qps: {no_constraints_workload_msg['qps']}")
                        print(f"pd_latency: {pd_latency}, no_constraints_latency: {no_constraints_latency}\n")
                        break
                    if pd_latency == 10e9 and no_constraints_latency == 10e9:
                        print(f"Both pd and no_constraints instance can not handdle the requests. {instance_deploy_msg}")
                        break
            else:
                # 降低no_constraints模型的qps，增加pd模型的qps
                while abs(pd_latency - no_constraints_latency) > 10 and no_constraints_latency > pd_latency:
                    if self.is_verbose:
                        print(f"Adjusting qps, pd_qps: {pd_workload_msg['qps']}, no_constraints_qps: {no_constraints_workload_msg['qps']}")
                        print(f"pd_latency: {pd_latency}, no_constraints_latency: {no_constraints_latency}\n")
                    # no_constraints_workload_msg['qps'] = no_constraints_qps * pd_latency / no_constraints_latency
                    no_constraints_workload_msg['qps'] = no_constraints_qps - 0.01
                    pd_workload_msg['qps'] = self._workload_msg['qps'] - no_constraints_workload_msg['qps']
                    if no_constraints_workload_msg['qps'] < 0:
                        pd_workload_msg['qps'] = self._workload_msg['qps']
                        pd_latency = self._get_pd_model_result(pd_workload_msg, pd_deployment_msg)
                        no_constraints_latency = 0
                        print(f"{instance_deploy_msg}: no_constraints_qps < 0, pd_latency:{pd_latency}({pd_workload_msg['qps']}), no_constraints_latency:{no_constraints_latency}({no_constraints_workload_msg['qps']})")
                        break
                    pd_latency = self._get_pd_model_result(pd_workload_msg, pd_deployment_msg)
                    no_constraints_latency = self._get_no_constraints_model_result(no_constraints_workload_msg, no_constraints_deployment_msg)
                    no_constraints_qps = no_constraints_workload_msg['qps']
                    counter += 1
                    if counter > 2000:
                        print(f"Adjust more than {counter} times")
                        print(f"Adjusting qps, pd_qps: {pd_workload_msg['qps']}, no_constraints_qps: {no_constraints_workload_msg['qps']}")
                        print(f"pd_latency: {pd_latency}, no_constraints_latency: {no_constraints_latency}\n")
                        break
                    if pd_latency == 10e9 and no_constraints_latency == 10e9:
                        print(f"Both pd and no_constraints instance can not handdle the requests. {instance_deploy_msg}")
                        break
            if pd_latency < 10e9 and no_constraints_latency < 10e9:
                return max(pd_latency, no_constraints_latency)
            return 10e9  # 无效解
        else:
            if is_verbose:
                print(f"Invalid placement: {instance_deploy_msg}")
            return 10e9  # 无效解
        
        
        

    def _get_placement_performance(self, states, is_verbose=False):
        '''
        Get the performance of a placement configuration.
        param states: The states of the cluster.
        return: The avg_request_time of the placement configuration.
        '''
        results = []
        for state in states:
            if self.is_verbose:
                print(f'state:{state}')
            placement = self._state_to_placement(state)
            if self.is_verbose:
                print(f"placement:{placement}")

            latency = self._get_model_result(self._model, self._workload_msg, placement, is_verbose=is_verbose)
            results.append(latency)
        return results
    
    def _constraints(self, states):
        constraint1 = []
        constraint2 = []
        for state in states:
            # 每个实例不跨节点 np.count_nonzero(state, axis=1) - 1 <= 0
            constraint1.append(sum(np.count_nonzero(state[:, 1:], axis=1) - 1))
            # pd分离时，Prefill和Decode实例至少有一个
            # 筛选出分配了GPU的实例
            assigned_instances = state[np.sum(state[:, 1:], axis=1) > 0]
            prefill_num = np.sum(assigned_instances[:, 0] == InstanceType.PREFILL)
            decode_num = np.sum(assigned_instances[:, 0] == InstanceType.DECODE)
            no_constraints_num = np.sum(assigned_instances[:, 0] == InstanceType.NO_CONSTRAINTS)

            if no_constraints_num > 0 and prefill_num == 0 and decode_num == 0:
                constraint2.append(-1)  # 满足约束
            else:
                constraint2.append(1 - min(prefill_num, decode_num))
            # if min(prefill_num, decode_num) < 1:
            #     print(f"Invalid placement: {state}, prefill_num: {prefill_num}, decode_num: {decode_num}")
        # print(f"constraint1: {constraint1}")
        # print(f"constraint2: {constraint2}")
        return np.column_stack([constraint1, constraint2])

    def _evaluate(self, states, out, *args, **kwargs):
        states = states.reshape(states.shape[0], *self._base_state.shape)
        avg_request_times = self._get_placement_performance(states)
        constraints = self._constraints(states)
        out["G"] = constraints
        out["F"] = np.column_stack([avg_request_times])

    def _crossover(self, states, **kwargs):
        states = states.reshape(*states.shape[:2], *self._base_state.shape)
        n_parents, n_matings, n_jobs, n_nodes = states.shape
        # Single-point crossover over jobs for all parent states.
        points = np.random.randint(n_jobs, size=(n_matings, 1))
        result = crossover_mask(states, np.arange(n_jobs) < points)
        
        return result.reshape(n_parents, n_matings, -1)

    def _mutation(self, states, **kwargs):
        states = states.reshape(states.shape[0], *self._base_state.shape)
        # (1) Randomly reset back to base state.
        mask = np.random.random(states.shape[:2]) < 0.1
        states = np.where(np.expand_dims(mask, 2), self._base_state, states)
        # (2) Randomly zero out some elements.
        prob = np.where(np.random.random(states.shape[:2]) < 0.1, 0.1, 0.0)
        states[np.random.random(states.shape) < np.expand_dims(prob, 2)] = 0
        # return states.reshape(states.shape[0], -1)
        # (3) Randomly change instance types
        for i in range(states.shape[0]):
            for j in range(states.shape[1]):
                if np.random.random() < 0.1:
                    states[i, j, 0] = (states[i, j, 0] + 1) % len(InstanceType)
        # (4) Randomly change number of GPUs. get value in [1,2,4]
                if np.random.random() < 0.2:
                    # states[i, j, 1:] = (states[i, j, 1:] + np.random.randint(1, self._num_gpus + 1, size=states.shape[2]-1)) % (self._num_gpus + 1)
                    states[i, j, 1:] = 0
                    random_node = np.random.randint(1, self._num_nodes + 1)
                    states[i, j, random_node] = np.random.choice([1,2,4], p=[0.4, 0.35, 0.25])
            # print(f"Mutated state {i}: {states[i]}")
        return states.reshape(states.shape[0], -1)

    def _repair(self, pop, **kwargs):
        states = pop.get("X")
        states = states.reshape(states.shape[0], *self._base_state.shape)

        # instance_types must be in {0, 1}(only pd).
        states[:, :, 0] = states[:, :, 0] % len(InstanceType)   # InstanceType 枚举值的数量
        # instance_gpus must be in {0, 1, ..., num_gpus}.
        states[:, :, 1:] = states[:, :, 1:] % (self._num_gpus + 1)

        # instance_gpus must be same for each type of instance.
        for i in range(states.shape[0]):
            # print(f"Before: {states[i]}")
            # 按实例类型分组
            unique_types = np.unique(states[i, :, 0])
            for t in unique_types:  # 遍历每种类型
                # 获取该类型所有实例的索引
                type_indices = np.where(states[i, :, 0] == t)[0]
                gpu_count = np.sum(states[i, type_indices, 1:], axis=1)
                if len(type_indices) > 1:
                    random_idx = np.random.choice(range(len(gpu_count)))
                    states[i, type_indices, 1:] = states[i, type_indices[random_idx], 1:]
            # print(f"After: {states[i]}")
        
        # The total gpu num that each node allocates to each instance cannot be greater than num_gpus.
        for i in range(states.shape[0]):
            allocated_gpus = np.sum(states[i, :, 1:], axis=0)
            for j, gpu_num in enumerate(allocated_gpus):
                if gpu_num > self._num_gpus:
                    # Randomly reset to 0.
                    while np.sum(states[i, :, 1+j]) > self._num_gpus:
                        random_instance = np.random.randint(states.shape[1])
                        states[i, random_instance, 1+j] = 0

        return pop.new("X", states.reshape(states.shape[0], -1))

## 2.2 Optimizer

In [ ]:


class Optimizer(object):
    def __init__(self, model, workload, num_gpus=4, num_nodes=1):
        '''
        Initialize a cluster with a given number of GPUs and nodes.
        param num_gpus: Number of GPUs per node.
        param num_nodes: Number of nodes in the cluster.
        '''
        self.model = model
        self.workload = workload
        self.num_gpus = num_gpus
        self.num_nodes = num_nodes
    
    def init_state(self):
        '''
        Initialize the cluster state.
        base_state: all instance types are NO_CONSTRAINTS, and only 1 GPU is allocated.
        return: The initialized cluster state.
        '''
        max_gpus = self.num_gpus * self.num_nodes   # 实例数最多可能达到的数量
        base_state = np.zeros((max_gpus,self.num_nodes+1), dtype=int)
        for i in range(max_gpus):
            base_state[i, 0] = InstanceType.PREFILL if i % 4 == 0 else InstanceType.DECODE
            base_state[i, 1 + (i // self.num_gpus)] = 1
        return base_state
    
    def _state_to_placement(self, state):
        '''
        Convert a cluster state to a placement configuration.
        param state: The state of the cluster.
        return: The placement configuration.
        '''
        # 暂时只考虑一个节点   只考虑一个节点: [(prefill_tps, decode_tps), (tps, [])]
        prefill_tps = []
        decode_tps = []
        tps = []
        for instance_idx in range(state.shape[0]):
            node_idx = np.where(state[instance_idx, 1:] > 0)[0]     # 找到该实例所在所有节点的索引
            node_gpus = state[instance_idx, 1:][node_idx]               # 找到该实例所在所有节点的GPU数
            if len(node_idx) == 0:
                continue
            if state[instance_idx, 0] == InstanceType.PREFILL:
                prefill_tps.append(sum(node_gpus))
            elif state[instance_idx, 0] == InstanceType.DECODE:
                decode_tps.append(sum(node_gpus))
            else:
                tps.append(sum(node_gpus))
        return [(prefill_tps, decode_tps), (tps, [])]
    
    def optimize(self, scale=None, is_verbose=False):
        '''
        Optimize the cluster configuration using a genetic algorithm.
        param base_state: The base state of the cluster.
        return: The optimized cluster configuration.
        '''
        base_state = self.init_state()
        
        states = np.expand_dims(base_state, 0)

        problem = Problem(self.model, self.workload, base_state, is_verbose)
        if scale is not None:
            problem.get_model_profiler_msg(scale)   # 调整model profile信息
            

        algorithm = NSGA2(
            pop_size=100, # 100
            # pymoo expects a flattened 2-D array.
            sampling=states.reshape(states.shape[0], -1),
            crossover=Crossover(),
            mutation=Mutation(),
            repair=Repair(),
        )
        result = pymoo.optimize.minimize(problem, algorithm, ("n_gen", 20), verbose=is_verbose)
        states = result.X.reshape(result.X.shape[0], self.num_gpus*self.num_nodes, self.num_nodes+1)
        print(f'Number of solutions found: {len(result.F)}')
        # for i in range(len(result.F)):
        #     print(f'Solution {i+1}: Objectives = {result.F[i]}, State = {states[i]}')
        #     placement = self._state_to_placement(states[i])
        #     print(f'Placement: {placement}')
        #     latnecy = problem._get_model_result(self.model, self.workload, placement, is_verbose=True)
        #     print(f'Rate model message(latnecy): {latnecy}\n')
        best_f = np.argmin(result.F)
        best_state = states[best_f]
        print(f'Best Solution: Objectives = {result.F[best_f]}, State = {best_state}')
        placement = self._state_to_placement(best_state)
        print(f'Placement: {placement}')
        latnecy = problem._get_model_result(self.model, self.workload, placement, is_verbose=True)
        print(f'Rate model message(latnecy): {latnecy}\n')
    def get_performance(self, state=None, scale={}, is_verbose=False):
        # base_state = np.array([
        #     [0,1],   # Prefill, 1 GPU (TP1) on node 1
        #     [1,1],   # Decode, 1 GPU (TP1) on node 1
        #     [2,1],   # No Constraints, 1 GPU (TP1) on node 1
        #     [2,1],
        #     [2,1],
        #     [2,1],
        #     [2,1],
        #     [2,1],
        # ])
        # base_state = np.array([
        #     [0,2],   # Prefill, 1 GPU (TP1) on node 1
        #     [1,2],   # Decode, 1 GPU (TP1) on node 1
        #     [2,2],   # No Constraints, 1 GPU (TP1) on node 1
        #     [2,2],
        #     [2,0],
        #     [2,0],
        #     [2,0],
        #     [2,0],
        # ])
        # base_state = np.array([
        #     [0,0],   # Prefill, 1 GPU (TP1) on node 1
        #     [0,2],   # Decode, 1 GPU (TP1) on node 1
        #     [1,2],   # No Constraints, 1 GPU (TP1) on node 1
        #     [2,0],
        #     [2,0],
        #     [2,0],
        #     [2,0],
        #     [2,0],
        # ])
        base_state = state
        problem = Problem(self.model, self.workload, base_state, is_verbose)
        problem.get_model_profiler_msg(scale)
        if len(base_state.shape) < 3:
            states = np.expand_dims(base_state, 0)
        else:
            states = base_state
        res = problem._get_placement_performance(states)
        print(res)

        return res



In [ ]:

state = np.array([[
        [0,0],   # Prefill, 1 GPU (TP1) on node 1
        [0,2],   # Decode, 1 GPU (TP1) on node 1
        [1,2],   # No Constraints, 1 GPU (TP1) on node 1
        [2,0],
        [2,0],
        [2,0],
        [2,0],
        [2,0],
    ],
    [
        [0,0],
        [0,0],
        [1,0],
        [2,2],
        [2,2],
        [2,0],
        [2,0],
        [2,0],
    ]])

model = 'llama-13b'
workload_msg = {
    'qps': 6,
    'prompt_len': 128, # 128, 182, 261
    'output_len': 128, # 256, 473, 491
    'is_ShareGPT': False,
}
optimizer = Optimizer(model=model, workload=workload_msg, num_gpus=4, num_nodes=1)
scale = {
    'migrateion_rate': 6,  # 直接指定迁移速率
    'migrateion': 0.0,     # 缩放
}
base_state = optimizer.optimize(scale)

## 2.3 测试


In [ ]:

state = np.array([[
        [0,1],   # Prefill, 1 GPU (TP1) on node 1
        [1,2],   # Decode, 1 GPU (TP1) on node 1
        [0,1],   # No Constraints, 1 GPU (TP1) on node 1
        [2,0],
        [2,0],
        [2,0],
        [2,0],
        [2,0],
    ],
    [
        [0,2],   # Prefill, 1 GPU (TP1) on node 1
        [1,2],   # Decode, 1 GPU (TP1) on node 1
        [2,0],   # No Constraints, 1 GPU (TP1) on node 1
        [2,0],
        [2,0],
        [2,0],
        [2,0],
        [2,0],
    ],
    [
        [0,0],   # Prefill, 1 GPU (TP1) on node 1
        [1,0],   # Decode, 1 GPU (TP1) on node 1
        [2,2],   # No Constraints, 1 GPU (TP1) on node 1
        [2,2],
        [2,0],
        [2,0],
        [2,0],
        [2,0],
    ]])
model = 'llama-13b'
workload_msg = {
    'qps': 4,
    'prompt_len': 128, # 128, 182, 261
    'output_len': 128, # 256, 473, 491
    'is_ShareGPT': False,
}
optimizer = Optimizer(model=model, workload=workload_msg, num_gpus=4, num_nodes=1)
scale = {
    # 'migrateion_rate': 6,  # 直接指定迁移速率
    # 'migrateion': 0.0,     # 缩放
}
latency = optimizer.get_performance(state=state, scale=scale, is_verbose=True)
# optimizer.optimize(is_verbose=True)


### 2.3.1 不同qps下的时延

In [ ]:

state = np.array([[
        [0,0],   # Prefill, 1 GPU (TP1) on node 1
        [0,2],   # Decode, 1 GPU (TP1) on node 1
        [1,2],   # No Constraints, 1 GPU (TP1) on node 1
        [2,0],
        [2,0],
        [2,0],
        [2,0],
        [2,0],
    ],
    [
        [0,0],
        [0,0],
        [1,0],
        [2,2],
        [2,2],
        [2,0],
        [2,0],
        [2,0],
    ]])
res = []
qps_all = range(1,20)
for qps in qps_all:
    model = 'llama-13b'
    workload_msg = {
        'qps': qps,
        'prompt_len': 256, # 128, 182, 261
        'output_len': 64, # 256, 473, 491
        'is_ShareGPT': False,
    }
    optimizer = Optimizer(model=model, workload=workload_msg, num_gpus=4, num_nodes=1)
    scale = {
        'migrateion_rate': qps,  # 直接指定迁移速率
        'migrateion': 0.0,     # 缩放
    }
    latency = optimizer.get_performance(state, scale, True)
    # optimizer.optimize()
    res.append(latency)
plt.plot(qps_all, [l[0] for l in res], label='2-2')
plt.plot(qps_all, [l[1] for l in res], label='2,2')
plt.xlabel('qps')
plt.ylabel('latency(ms)')
plt.legend()
plt.title('Request Latency(Input Len: 512, Output Len: 64)')
plt.show()

### 2.3.2 不同迁移速率下的时延

In [ ]:

state = np.array([[
        [0,0],   # Prefill, 1 GPU (TP1) on node 1
        [0,2],   # Decode, 1 GPU (TP1) on node 1
        [1,2],   # No Constraints, 1 GPU (TP1) on node 1
        [2,0],
        [2,0],
        [2,0],
        [2,0],
        [2,0],
    ],
    [
        [0,0],
        [0,0],
        [1,0],
        [2,2],
        [2,2],
        [2,0],
        [2,0],
        [2,0],
    ]])
res = []
for m_rate in range(1,10):
    model = 'llama-13b'
    workload_msg = {
        'qps': 4,
        'prompt_len': 512, # 128, 182, 261
        'output_len': 64, # 256, 473, 491
        'is_ShareGPT': False,
    }
    optimizer = Optimizer(model=model, workload=workload_msg, num_gpus=4, num_nodes=1)
    scale = {
        'migrateion_rate': m_rate,  # 直接指定迁移速率
        'migrateion': 0.0,     # 缩放
    }
    latency = optimizer.get_performance(state, scale, False)
    optimizer.optimize()
    res.append(latency)
plt.plot(range(1,10), [l[0] for l in res], label='2-2')
plt.plot(range(1,10), [l[1] for l in res], label='2,2')
plt.xlabel('migrateion_rate')
plt.ylabel('latency(ms)')
plt.legend()
plt.title('Request Latency(Input Len: 512, Output Len: 64)')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
arr = np.arange(0, 5.1, 0.1)  # 终止值需略超过5（如5.1）以确保包含5
y = [-x + (x*x+10)**0.5 for x in arr]
plt.plot(arr, y)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Plot of y = -x + sqrt(x^2 + 1000)')
plt.grid()
plt.show()